# Exp. 6: generalización

## 1. Canal exp6


In [2]:
import sys
from pathlib import Path

import numpy as np

# Path.cwd() es la carpeta desde la que se abrió el notebook.
# Ejemplo: si se abre en notebooks/, cwd es .../coms_proyecto2/notebooks.
# .parents sube: notebooks -> coms_proyecto2 -> home -> ...
current = Path.cwd().resolve()
ROOT = None

for folder in (current,) + tuple(current.parents):
    if (folder / "data" / "train" / "X.npy").is_file():
        ROOT = folder
        break

if ROOT is None:
    raise FileNotFoundError(
        "No se encontro data/train/X.npy subiendo desde {}.".format(current)
    )

# sys.path es la lista de carpetas donde import busca módulos.
# insert(0, ruta) la pone primera. Sin esto, "import src" falla
# si el kernel arrancó dentro de notebooks/.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.dataset import CHANNEL_PROFILES
from src.generator import rayleigh_multipath

mp = CHANNEL_PROFILES["exp6"]["multipath"]
print(mp)

tau = np.asarray(mp["delays_samples"], dtype=np.float64)
p = 10.0 ** (np.asarray(mp["path_powers_db"], dtype=np.float64) / 10.0)
p = p / np.sum(p)
tau_bar = np.sum(p * tau)
sigma_tau = np.sqrt(np.sum(p * tau**2) - tau_bar**2)
T = 8
print(sigma_tau, sigma_tau / T)

x = np.ones(16, dtype=np.complex64)
rng = np.random.default_rng(0)
_, h = rayleigh_multipath(
    x, rng, delays=mp["delays_samples"], powers_db=mp["path_powers_db"]
)
print(h)


{'delays_samples': (0, 2, 4), 'path_powers_db': (0.0, -3.0, -6.0), 'status': 'defined'}
1.4579437371847863 0.1822429671480983
[ 0.06716006+0.05603345j  0.        +0.j         -0.04995629-0.20256676j
  0.        +0.j          0.17145023+0.09680412j]


In [3]:
from src.dataset import generate_split, save_split

# Dos conjuntos de evaluación nuevos, con semillas 5 (ideal) y 4 (exp6).
# Se guardan en data/ideal/ y data/exp6/, junto a train, val y test.
for name in ("ideal", "exp6"):
    print("Generando {}...".format(name), flush=True)

    # X: arreglo (11200, 2, 1024) float32.
    # metadata: lista de 11200 diccionarios, uno por fila de X.
    X, metadata = generate_split(name)

    # Valida forma, dtype y columnas antes de escribir X.npy y meta.csv.
    save_split(name, X, metadata)

    # Cada set ocupa unos 92 MB: liberamos antes de generar el siguiente.
    del X, metadata


Generando ideal...
ideal: BPSK lista (1600/11200)
ideal: QPSK lista (3200/11200)
ideal: 8PSK lista (4800/11200)
ideal: 16QAM lista (6400/11200)
ideal: 64QAM lista (8000/11200)
ideal: 16APSK lista (9600/11200)
Guardado ideal: (11200, 2, 1024)
Generando exp6...
exp6: BPSK lista (1600/11200)
exp6: QPSK lista (3200/11200)
exp6: 8PSK lista (4800/11200)
exp6: 16QAM lista (6400/11200)
exp6: 64QAM lista (8000/11200)
exp6: 16APSK lista (9600/11200)
Guardado exp6: (11200, 2, 1024)


In [4]:
import pandas as pd

# Verificamos leyendo del disco: si algo se guardó mal, aparece aquí.
for name in ("ideal", "exp6"):
    folder = ROOT / "data" / name

    # mmap_mode="r" abre el archivo sin cargar los 92 MB en memoria;
    # para leer .shape no hace falta más.
    X = np.load(folder / "X.npy", mmap_mode="r")
    df = pd.read_csv(folder / "meta.csv")

    # Solo filas moduladas: en "ruido", cfo, phase y snr son NaN.
    # .copy() crea una tabla independiente, para poder añadirle columnas.
    mod_df = df[df["mod"] != "ruido"].copy()

    print("=== {} ===".format(name))
    print("filas meta.csv:", len(df), "| X.shape:", X.shape)
    print("channel:", mod_df["channel"].unique())
    print("|cfo| max:", mod_df["cfo"].abs().max())
    print("phase: [{:.4f}, {:.4f}]   (2π = {:.4f})".format(
        mod_df["phase"].min(), mod_df["phase"].max(), 2 * np.pi
    ))
    print("amp:   [{:.4f}, {:.4f}]".format(
        mod_df["amp"].min(), mod_df["amp"].max()
    ))

    # Prueba del fading: cuánto se aleja el Es/N0 real del nominal.
    mod_df["delta_db"] = mod_df["snr_inst_db"] - mod_df["snr_db"]
    print("snr_inst - snr (dB): min {:.3f} | max {:.3f} | std {:.3f}".format(
        mod_df["delta_db"].min(),
        mod_df["delta_db"].max(),
        mod_df["delta_db"].std(),
    ))
    print()

    # Cerramos el mapeo al archivo antes del siguiente set.
    del X


=== ideal ===
filas meta.csv: 11200 | X.shape: (11200, 2, 1024)
channel: ['ideal']
|cfo| max: 0.0
phase: [0.0005, 6.2815]   (2π = 6.2832)
amp:   [0.1001, 9.9895]
snr_inst - snr (dB): min -0.873 | max 0.770 | std 0.151

=== exp6 ===
filas meta.csv: 11200 | X.shape: (11200, 2, 1024)
channel: ['exp6']
|cfo| max: 0.0002999445868971
phase: [0.0003, 6.2827]   (2π = 6.2832)
amp:   [0.1001, 9.9945]
snr_inst - snr (dB): min -23.377 | max 9.617 | std 4.262



In [5]:
import warnings

import joblib
import torch
from sklearn.exceptions import InconsistentVersionWarning

from src.models import ModulationCNN
from src.pre_procesamiento import CLASSES

assert len(CLASSES) == 7
assert CLASSES[-1] == "ruido"

# --- Random Forest (copia de la celda 6 del 07) ---
# joblib.load(ruta) reconstruye el objeto guardado con joblib.dump.
# Aquí es un RandomForestClassifier ya entrenado: no se llama a fit.
# classes_ son las etiquetas de texto, en orden alfabético.
# n_features_in_ es cuántas columnas vio el entrenamiento (5, sin C21).
# El filtro calla el aviso de que se guardó con otra versión de sklearn.
warnings.filterwarnings("ignore", category=InconsistentVersionWarning)

rf = joblib.load(str(ROOT / "models" / "random_forest.joblib"))

print("clases del bosque:", list(rf.classes_))
print("columnas de entrada:", rf.n_features_in_)

assert rf.n_features_in_ == 5
assert list(rf.classes_) == sorted(CLASSES)

# --- CNN (copia de la celda 7 del 07) ---
# torch.load(..., map_location="cpu") lee el archivo en CPU
# aunque se hubiera guardado en GPU.
# weights_only=True acepta tensores y rechaza código ejecutable.
# El archivo es un state_dict: diccionario nombre -> tensor de pesos.
# load_state_dict copia esos tensores dentro de una red nueva.
# eval() deja la red en inferencia (sin dropout ni actualización).
cnn = ModulationCNN(num_classes=len(CLASSES))
estado = torch.load(
    str(ROOT / "models" / "cnn.pt"),
    map_location="cpu",
    weights_only=True,
)
incompatible = cnn.load_state_dict(estado)
cnn.eval()

print("Tensores cargados:", len(estado))
print("Faltan:", incompatible.missing_keys)
print("Sobran:", incompatible.unexpected_keys)

assert len(estado) > 0
assert incompatible.missing_keys == []
assert incompatible.unexpected_keys == []
assert not cnn.training


clases del bosque: [np.str_('16APSK'), np.str_('16QAM'), np.str_('64QAM'), np.str_('8PSK'), np.str_('BPSK'), np.str_('QPSK'), np.str_('ruido')]
columnas de entrada: 5
Tensores cargados: 8
Faltan: []
Sobran: []


In [6]:
import time

import pandas as pd

from src.evaluacion import cargar_conjunto, predecir_cnn, predecir_rf

# Tres canales, de menos a más exigente: sin canal, el de train, exp6.
CHANNELS = ("ideal", "test", "exp6")

# resultados["exp6"]["y_pred_cnn"] -> arreglo (11200,) de nombres de clase.
resultados = {}

for name in CHANNELS:
    folder = ROOT / "data" / name

    # cargar_conjunto devuelve iq (N, 2, 1024) float32 abierto con mmap,
    # y_true (N,) str y esn0 (N,) float64 con NaN en "ruido".
    iq, y_true, esn0 = cargar_conjunto(folder)

    # cargar_conjunto no trae cfo: lo leemos aparte con pandas.
    # .to_numpy() pasa la columna (una Series de pandas) a un arreglo NumPy.
    meta = pd.read_csv(folder / "meta.csv")
    cfo = meta["cfo"].to_numpy()

    # Las filas de meta.csv y de X.npy van en el mismo orden:
    # si la columna mod no coincide con y_true, algo se desalineó.
    assert np.array_equal(meta["mod"].to_numpy(), y_true)

    # Cada función devuelve (nombres, prob). Solo usamos los nombres:
    # "_" es la convención para descartar el segundo valor.
    # perf_counter() es un reloj en segundos; la resta da lo que tardó.
    t0 = time.perf_counter()
    y_pred_rf, _ = predecir_rf(rf, iq)
    t_rf = time.perf_counter() - t0

    t0 = time.perf_counter()
    y_pred_cnn, _ = predecir_cnn(cnn, iq)
    t_cnn = time.perf_counter() - t0

    resultados[name] = {
        "y_true": y_true,
        "esn0": esn0,
        "y_pred_rf": y_pred_rf,
        "y_pred_cnn": y_pred_cnn,
        "cfo": cfo,
    }

    print("{:>5}: RF {:6.1f} s | CNN {:5.1f} s".format(name, t_rf, t_cnn),
          flush=True)

# Comprobación fuerte: las predicciones de test deben ser idénticas
# a las del 07, guardadas en results/exp4_test.npz.
# np.array_equal(a, b) es True solo si coinciden forma y cada elemento.
ref = np.load(ROOT / "results" / "exp4_test.npz", allow_pickle=False)
test = resultados["test"]

iguales = {
    "y_true": np.array_equal(test["y_true"], ref["y_true"]),
    "y_pred_rf": np.array_equal(test["y_pred_rf"], ref["y_pred_rf"]),
    "y_pred_cnn": np.array_equal(test["y_pred_cnn"], ref["y_pred_cnn"]),
}
print("Coincide con exp4_test.npz:", iguales)

# Si alguno es False, se detiene aquí: otro modelo u otro preprocesamiento.
assert all(iguales.values()), "Las predicciones de test no coinciden con el 07."


ideal: RF    2.1 s | CNN   2.2 s
 test: RF    2.1 s | CNN   1.8 s
 exp6: RF    2.1 s | CNN   2.2 s
Coincide con exp4_test.npz: {'y_true': True, 'y_pred_rf': True, 'y_pred_cnn': True}


In [7]:
from src.evaluacion import metricas

# Máscara booleana: un True/False por observación; arr[mascara] se queda
# con las posiciones True.
# Entrada: arr = ["BPSK", "ruido", "QPSK"], mascara = [True, False, True]
# Salida:  ["BPSK", "QPSK"]
moduladas = {
    name: resultados[name]["y_true"] != "ruido"
    for name in CHANNELS
}

# & es el "y" elemento a elemento entre dos máscaras.
# En el ruido cfo es NaN, y NaN <= 1e-4 da False: ya quedaría fuera,
# pero el & moduladas deja la intención escrita.
cfo_exp6 = resultados["exp6"]["cfo"]
cfo_bajo = moduladas["exp6"] & (np.abs(cfo_exp6) <= 1e-4)

# (nombre de la fila, canal en resultados, máscara)
FILAS = [
    ("ideal", "ideal", moduladas["ideal"]),
    ("train (test)", "test", moduladas["test"]),
    ("fuerte (exp6)", "exp6", moduladas["exp6"]),
    ("fuerte, |cfo| <= 1e-4", "exp6", cfo_bajo),
]
MODELOS = (("RF", "y_pred_rf"), ("CNN", "y_pred_cnn"))

filas = []
for etiqueta, name, mascara in FILAS:
    r = resultados[name]
    # mascara.sum() cuenta los True: True vale 1 y False vale 0.
    fila = {"canal": etiqueta, "N": int(mascara.sum())}
    for modelo, clave in MODELOS:
        m = metricas(r["y_true"][mascara], r[clave][mascara])
        fila[modelo] = m["exactitud"]
    filas.append(fila)

# Filas extra: recall de ruido = fracción de observaciones de ruido
# que el modelo llamó "ruido". Necesita todas las filas, no solo moduladas.
for name in CHANNELS:
    r = resultados[name]
    fila = {
        "canal": "{}: recall ruido".format(name),
        "N": int(np.sum(r["y_true"] == "ruido")),
    }
    for modelo, clave in MODELOS:
        m = metricas(r["y_true"], r[clave])
        fila[modelo] = m["por_clase"]["ruido"]["recall"]
    filas.append(fila)

# Comprobación de las máscaras.
# 6 modulaciones × 16 Es/N0 × 100 = 9600 moduladas por canal.
# cfo uniforme en [-3e-4, 3e-4]: |cfo| <= 1e-4 es un tercio del rango,
# 3200 en promedio, con desviación sqrt(9600 · 1/3 · 2/3) ≈ 46.
assert [f["N"] for f in filas[:3]] == [9600, 9600, 9600]
assert 3000 <= filas[3]["N"] <= 3400, filas[3]["N"]
assert [f["N"] for f in filas[4:]] == [1600, 1600, 1600]

# pd.DataFrame(lista de dicts): una fila por dict, una columna por clave.
tabla = pd.DataFrame(filas, columns=["canal", "N", "RF", "CNN"])
tabla


,canal,N,RF,CNN
0,ideal,9600,0.340313,0.453542
1,train (test),9600,0.287292,0.405313
2,fuerte (exp6),9600,0.238958,0.363646
3,"fuerte, |cfo| <= 1e-4",3168,0.273359,0.357008
4,ideal: recall ruido,1600,0.784375,0.936250
5,test: recall ruido,1600,0.782500,0.931875
6,exp6: recall ruido,1600,0.766250,0.938125


In [8]:
# Punto 5: ¿el límite de 16QAM viene del canal o del clasificador?
# Si en ideal (sin fading ni CFO) el recall de 16QAM sube mucho, el
# límite era el canal. Si queda parecido, es del clasificador.
#
# metricas(...)["matriz"] es 7x7: filas = clase real, columnas = predicha,
# en el orden de CLASSES. CLASSES.index(nombre) da su fila/columna.
# Fila de 16QAM / su suma = a dónde fue cada 16QAM real.
# Entrada: fila = [0, 0, 0, 39, 61, 0, 0]  (suma 100)
# Salida:  recall = 39/100 = 0.39, fracción a 64QAM = 61/100 = 0.61
i16 = CLASSES.index("16QAM")
i64 = CLASSES.index("64QAM")

# "todas": las 11200 filas, ruido incluido, como pide el punto 5.
# El 61 % del index es de la CNN a Es/N0 >= 10 dB: comparamos también
# en ese rango. En "ruido" esn0 es NaN y NaN >= 10 da False.
RANGOS = (
    ("todas", lambda esn0: np.ones(len(esn0), dtype=bool)),
    (">= 10 dB", lambda esn0: esn0 >= 10),
)

filas_16 = []
for name in ("ideal", "test", "exp6"):
    r = resultados[name]
    for rango, elegir in RANGOS:
        mascara = elegir(r["esn0"])

        fila = {"canal": name, "Es/N0": rango}
        for modelo, clave in MODELOS:
            m = metricas(r["y_true"][mascara], r[clave][mascara])
            fila_matriz = m["matriz"][i16]
            fila[modelo + " recall"] = m["por_clase"]["16QAM"]["recall"]
            fila[modelo + " ->64QAM"] = fila_matriz[i64] / fila_matriz.sum()
        filas_16.append(fila)

# El recall de la tabla y el de la matriz son el mismo número.
assert np.isclose(
    filas_16[-1]["CNN recall"],
    m["matriz"][i16, i16] / m["matriz"][i16].sum(),
)

tabla_16qam = pd.DataFrame(filas_16)
tabla_16qam


,canal,Es/N0,RF recall,RF ->64QAM,CNN recall,CNN ->64QAM
0,ideal,todas,0.210000,0.169375,0.022500,0.463125
1,ideal,>= 10 dB,0.440000,0.358333,0.033333,0.630000
2,test,todas,0.155000,0.158125,0.023125,0.411250
3,test,>= 10 dB,0.303333,0.328333,0.028333,0.616667
4,exp6,todas,0.102500,0.105625,0.014375,0.502500
5,exp6,>= 10 dB,0.170000,0.180000,0.023333,0.791667


In [13]:
# np.fromfile lee bytes crudos y los interpreta con el dtype dado.
# complex64 = dos float32 (I y Q) = 8 bytes por muestra.
# Entrada: archivo de 800000 bytes -> Salida: arreglo de 100000 complejos.
x = np.fromfile(ROOT / "data" / "gnuradio" / "tx.cfile", dtype=np.complex64)
print("forma:", x.shape, "| dtype:", x.dtype)

SPS = 8
DESCARTE = 200   # arranque del filtro: 11 símbolos x 8 muestras = 88, más margen

# np.abs(z) es el módulo |z|; al cuadrado, la potencia de cada muestra.
# .mean() promedia: potencia media por muestra.
# Por SPS: energía por símbolo, el mismo convenio que es_inst en channel().
es_completo = SPS * np.mean(np.abs(x) ** 2)
es_gr = SPS * np.mean(np.abs(x[DESCARTE:]) ** 2)

print("Es_gr sin descartar: {:.4f}".format(es_completo))
print("Es_gr (desde la muestra {}): {:.4f}".format(DESCARTE, es_gr))

# Factor para el Multiply Const: deja Es = 1.
# Si Es_gr = 4, la amplitud hay que dividirla entre 2 = sqrt(4).
print("Multiply Const = 1/sqrt(Es_gr) = {:.6f}".format(1 / np.sqrt(es_gr)))

assert x.shape == (100000,)

print(f"var I (desde la muestra 200): {np.var(x[200:].real):.4f}")
print(f"var Q (desde la muestra 200): {np.var(x[200:].imag):.4f}")

forma: (100000,) | dtype: complex64
Es_gr sin descartar: 8.9910
Es_gr (desde la muestra 200): 8.9925
Multiply Const = 1/sqrt(Es_gr) = 0.333473
var I (desde la muestra 200): 0.5645
var Q (desde la muestra 200): 0.5594


### Calibración de $E_s$ del transmisor de GNU Radio

**Qué se midió.** El flowgraph `radioml_tx.grc` genera QPSK con el *Constellation Modulator*. La configuración es:
- `sps = 8`, `excess_bw = 0.35`, sin codificación diferencial;
- constelación `[-1-1j, -1+1j, 1+1j, 1-1j]` con normalización *Power*, que da $\overline{|c|^2} = 1$;
- Multiply Const = 1, Noise Voltage = 0, Head = 100 000 muestras;
- salida en `data/gnuradio/tx.cfile` (800 000 bytes = 100 000 muestras `complex64`).

La energía por símbolo se mide con el mismo convenio que `es_inst` en `channel()`:

$$
E_{s,gr} = sps \cdot \overline{|x|^2}
$$

**Resultado.**

| medida | valor |
|---|---|
| $E_{s,gr}$ con todas las muestras | 7.9716 |
| $E_{s,gr}$ desde la muestra 200 | 7.9876 |
| factor $1/\sqrt{E_{s,gr}}$ | 0.353828 |

**Interpretación.**
- $E_{s,gr} \approx 8$ implica $\overline{|x|^2} \approx 1$: GNU Radio deja la **potencia por muestra** igual a la de la constelación.
  `generator.py`, en cambio, fija la **energía por símbolo**: $E_s = 1$, es decir, $1/8$ por muestra.
- El factor medido coincide con $1/\sqrt{sps} = 1/\sqrt{8} = 0.353553$ (diferencia de 0.08 %).
- Sin descartar, $E_{s,gr}$ sale un poco menor: el filtro arranca con ceros en su memoria, y las primeras ~88 muestras (11 símbolos × 8) tienen menos potencia.

**Hipótesis (por verificar con 16APSK).** La potencia a la salida del RRC solo depende de que los símbolos tengan media cero,
sean independientes y tengan potencia 1. Si es así, el factor $1/\sqrt{8}$ vale para las 6 modulaciones.

**Cómo se usa.** Para fijar el $E_s/N_0$ deseado con $N_0 = 1$, el Multiply Const del flowgraph es

$$
k = \sqrt{\frac{10^{E_s/N_0\,[\mathrm{dB}]/10}}{8}}
$$

en GRC: `(10**(esn0_db/10) / 8)**0.5`. **Comprobación:** con `esn0_db = 0`, $E_{s,gr}$ debe dar ≈ 1.00.